In [1]:



import pandas as pd
import plotly.express as px
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import statsmodels.api as sm

In [2]:


def load_data(file_path):
    """
    Load data from a CSV file into a pandas DataFrame.

    Parameters:
    file_path (str): The path to the CSV file.

    Returns:
    pd.DataFrame: A DataFrame containing the loaded data.
    """
    try:
        data = pd.read_csv(file_path)
        return data
    except FileNotFoundError:
        print(f"Error: The file at {file_path} was not found.")
        return None
    except pd.errors.EmptyDataError:
        print("Error: The file is empty.")
        return None
    except pd.errors.ParserError:
        print("Error: There was a parsing error while reading the file.")
        return None

df_earthquakes = load_data('../earthquake_1995-2023.csv')

print (df_earthquakes.head())
df_earthquakes.info()






                                      title  magnitude         date_time  cdi  \
0          M 6.5 - 42 km W of Sola, Vanuatu        6.5  16-08-2023 12:47    7   
1  M 6.5 - 43 km S of Intipucá, El Salvador        6.5  19-07-2023 00:22    8   
2  M 6.6 - 25 km ESE of Loncopué, Argentina        6.6  17-07-2023 03:05    7   
3     M 7.2 - 98 km S of Sand Point, Alaska        7.2  16-07-2023 06:48    6   
4                  M 7.3 - Alaska Peninsula        7.3  16-07-2023 06:48    0   

   mmi   alert  tsunami  sig net  nst      dmin    gap magType    depth  \
0    4   green        0  657  us  114  7.177000   25.0     mww  192.955   
1    6  yellow        0  775  us   92  0.679000   40.0     mww   69.727   
2    5   green        0  899  us   70  1.634000   28.0     mww  171.371   
3    6   green        1  860  us  173  0.907000   36.0     mww   32.571   
4    5     NaN        1  820  at   79  0.879451  172.8      Mi   21.000   

   latitude  longitude               location      continent  

Data quality
- date_time column is defined as str and should be updated to date time for a better analysis
- the columns alert, location, continent and country have absent values. This information wont be deleted to analyze all information


In [3]:
df_earthquakes['date_time'] = pd.to_datetime(df_earthquakes['date_time'], errors='coerce')

df_earthquakes.info()


<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 19 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   title      1000 non-null   str           
 1   magnitude  1000 non-null   float64       
 2   date_time  1000 non-null   datetime64[us]
 3   cdi        1000 non-null   int64         
 4   mmi        1000 non-null   int64         
 5   alert      449 non-null    str           
 6   tsunami    1000 non-null   int64         
 7   sig        1000 non-null   int64         
 8   net        1000 non-null   str           
 9   nst        1000 non-null   int64         
 10  dmin       1000 non-null   float64       
 11  gap        1000 non-null   float64       
 12  magType    1000 non-null   str           
 13  depth      1000 non-null   float64       
 14  latitude   1000 non-null   float64       
 15  longitude  1000 non-null   float64       
 16  location   994 non-null    str           
 17  contine

C:\Users\car_f\AppData\Local\Temp\ipykernel_18296\1809202310.py:1: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_earthquakes['date_time'] = pd.to_datetime(df_earthquakes['date_time'], errors='coerce')


In [4]:


# 1. Limpieza de datos general
df_clean = df_earthquakes.dropna(subset=['magnitude', 'cdi', 'depth']).copy()

# Función auxiliar para obtener línea de tendencia (OLS)
def get_ols_line(x_data, y_data):
    X = sm.add_constant(x_data)
    model = sm.OLS(y_data, X).fit()
    x_range = np.linspace(x_data.min(), x_data.max(), 100)
    y_pred = model.predict(sm.add_constant(x_range))
    return x_range, y_pred

# 2. Crear lienzo de 1 fila y 2 columnas
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Intensidad Reportada (CDI) vs. Magnitud',
        'Profundidad (km) vs. Magnitud'
    ),
    horizontal_spacing=0.1
)

# --- GRÁFICA 1: CDI vs Magnitud ---
# Puntos dispersos
fig.add_trace(
    go.Scatter(
        x=df_clean['cdi'],
        y=df_clean['magnitude'],
        mode='markers',
        marker=dict(color='#2b5c8f', opacity=0.5, size=7),
        name='Sismos (CDI)',
        customdata=df_clean[['title', 'location', 'country', 'depth']],
        hovertemplate='<b>%{customdata[0]}</b><br>Ubicación: %{customdata[1]}, %{customdata[2]}<br>CDI: %{x}<br>Magnitud: %{y}<br>Profundidad: %{customdata[3]} km<extra></extra>'
    ),
    row=1, col=1
)

# Línea de tendencia OLS (Gráfica 1)
x_ols1, y_ols1 = get_ols_line(df_clean['cdi'], df_clean['magnitude'])
fig.add_trace(
    go.Scatter(
        x=x_ols1, y=y_ols1,
        mode='lines',
        line=dict(color='#d9534f', width=2),
        name='Tendencia OLS (CDI)'
    ),
    row=1, col=1
)

# --- GRÁFICA 2: Profundidad vs Magnitud ---
# Puntos dispersos (con escala de color Viridis por magnitud)
fig.add_trace(
    go.Scatter(
        x=df_clean['depth'],
        y=df_clean['magnitude'],
        mode='markers',
        marker=dict(
            color=df_clean['magnitude'],
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title='Magnitud', x=1.02, len=0.8),
            opacity=0.6,
            size=7
        ),
        name='Sismos (Profundidad)',
        customdata=df_clean[['title', 'location', 'country']],
        hovertemplate='<b>%{customdata[0]}</b><br>Ubicación: %{customdata[1]}, %{customdata[2]}<br>Profundidad: %{x} km<br>Magnitud: %{y}<extra></extra>'
    ),
    row=1, col=2
)

# Línea de tendencia OLS (Gráfica 2)
x_ols2, y_ols2 = get_ols_line(df_clean['depth'], df_clean['magnitude'])
fig.add_trace(
    go.Scatter(
        x=x_ols2, y=y_ols2,
        mode='lines',
        line=dict(color='#d9534f', width=2),
        name='Tendencia OLS (Profundidad)'
    ),
    row=1, col=2
)

# 3. Homogeneizar estilo del Layout
fig.update_layout(
    title_text='Análisis Comparativo de Magnitud Sísmica',
    template='plotly_white',
    height=550,
    width=1200,
    showlegend=False,
    font=dict(size=12)
)

# Configuración de títulos de ejes
fig.update_xaxes(title_text='Intensidad CDI (Mercalli Modificada)', row=1, col=1)
fig.update_yaxes(title_text='Magnitud (Escala Richter)', row=1, col=1)

fig.update_xaxes(title_text='Profundidad del Hipocentro (km)', row=1, col=2)
fig.update_yaxes(title_text='Magnitud (Escala Richter)', row=1, col=2)

# Mostrar la figura unificada
fig.show()

La magnitud mide la energía liberada en la fuente y es independiente de la profundidad o de la presencia humana.

El CDI mide el impacto percibido, por lo que depende fuertemente de la proximidad a centros urbanos y la profundidad del evento (los sismos superficiales se sienten con mayor fuerza en la superficie que los profundos de igual magnitud).

En un modelo predictivo o cuantitativo, la profundidad y la ubicación (distancia a población) actuarían como variables mediadoras para explicar la severidad de la intensidad reportada (CDI).

In [5]:
# Limpiar filas sin dato de magnitud
df_clean = df_earthquakes.dropna(subset=['magnitude'])

# Crear el histograma de magnitudes
fig = px.histogram(
    df_clean,
    x='magnitude',
    nbins=30,
    title='Distribución de la Magnitud de los Terremotos',
    labels={'magnitude': 'Magnitud'},
    color_discrete_sequence=['#1f77b4'],
    template='plotly_white'
)

# Personalizar ejes y separación entre barras
fig.update_layout(
    xaxis_title='Magnitud (Escala Richter / Momento)',
    yaxis_title='Frecuencia (Cantidad de Terremotos)',
    bargap=0.05,
    font=dict(size=12)
)

# Mostrar el gráfico interactivo
fig.show()

Se ilustra perfectamente la Ley de Gutenberg-Richter. Esta ley fundamental de la sismología establece que existe una relación exponencial inversa entre la magnitud de un terremoto y la frecuencia con la que ocurre.